<a href="https://colab.research.google.com/github/mahanthianuradha/Player-score-calculator/blob/master/EmployeeAnalysis_AnuradhaManti_iitp_aimlt_2601949_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

NAME : ANURADHA MANTI
ID   :iitp_aimlt_2601949

Task 1: Create Database
Write Python code to create employees.db with an employees table containing at least 40 records:

emp_id (INTEGER): 1, 2, 3...
name (TEXT): Employee_1, Employee_2...
department (TEXT): Engineering, Sales, Marketing, HR, Finance
salary (REAL): Random between 50,000-150,000
years_experience (INTEGER): Random between 1-15
performance_score (REAL): Random between 1.0-5.0
Use sqlite3 and random modules to generate realistic data.



> Add blockquote



In [65]:
import pandas as pd
import random
import sqlite3
conn = sqlite3.connect('employees.db')
conn.close()

In [66]:
conn = sqlite3.connect('employees.db')
cursor = conn.cursor()
create_table_query = '''
CREATE TABLE IF NOT EXISTS employees (
    emp_id INTEGER PRIMARY KEY AUTOINCREMENT ,
    name TEXT,
    department TEXT,
    salary REAL,
    years_experience INTEGER,
    performance_score REAL
    );
    '''

cursor.execute(create_table_query)


conn.commit()


query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql(query,conn)
print(tables)

query = 'PRAGMA table_info(employees)'
columns_info = pd.read_sql(query,conn)
print(columns_info)



              name
0        employees
1  sqlite_sequence
   cid               name     type  notnull dflt_value  pk
0    0             emp_id  INTEGER        0       None   1
1    1               name     TEXT        0       None   0
2    2         department     TEXT        0       None   0
3    3             salary     REAL        0       None   0
4    4   years_experience  INTEGER        0       None   0
5    5  performance_score     REAL        0       None   0


In [67]:
employees_data=[]
#create data for 50 employees using for loop
for i in range (1,51):
  #print (i)
  employee_name = f'Employee_{i}'
  department_name = random.choice(['Engineering','Sales', 'Marketing', 'HR', 'Finance'])
  salary = random.randint(50000 , 150000)
  experience = random.randint(1 , 15)
  score = random.uniform(1.0 , 5.0)
  #print(employee_name,department_name,salary,experience,score)
  #create row using column values for each employee
  row = (employee_name,department_name,salary,experience,score)
  #add row to employee data
  employees_data.append(row)


cursor.executemany('''
INSERT INTO employees(name,department, salary,years_experience, performance_score)
VALUES(?,?,?,?,?)
''',employees_data
)

conn.commit()

query = "SELECT * FROM employees;"
cursor.execute(query)
print(len(cursor.fetchall()))


150


Task 2: SQL Queries
Write SQL queries and load results into Pandas DataFrames:

Query 1: Employees with performance_score >= 4.0 AND years_experience >= 3. Show name, department, salary, performance_score. Sort by performance_score DESC. Limit 15.

Query 2: Employees with salary BETWEEN
70,000AND110,000 in Engineering OR Sales. Show all columns. Sort by department, then salary DESC.

Query 3: Count employees per department and calculate average salary per department. Sort by avg_salary DESC.

In [58]:
query = '''SELECT  name, department, salary, performance_score
           FROM employees WHERE
           performance_score >= 4.0 AND years_experience >= 3
           ORDER BY performance_score DESC LIMIT 15
           '''
df = pd.read_sql (query , conn)
print(df)

query = '''SELECT * FROM employees
           WHERE salary BETWEEN 70000 AND 110000
           AND (department = 'Engineering'  OR department = 'Sales')
           order by department desc, salary
           '''
df = pd.read_sql (query , conn)
print(df)

           name   department    salary  performance_score
0   Employee_10  Engineering   53711.0           4.956407
1   Employee_48    Marketing   70440.0           4.877628
2    Employee_1    Marketing  129396.0           4.863769
3    Employee_7    Marketing  121715.0           4.858985
4   Employee_20        Sales   81006.0           4.720148
5   Employee_23           HR  122379.0           4.663215
6   Employee_35    Marketing   51820.0           4.615563
7   Employee_16  Engineering   75767.0           4.605584
8   Employee_36      Finance  106095.0           4.580189
9   Employee_43  Engineering  115779.0           4.579032
10  Employee_42           HR   56162.0           4.540625
11  Employee_42      Finance   67685.0           4.508197
12  Employee_15           HR   64647.0           4.407723
13   Employee_9      Finance   91536.0           4.391741
14  Employee_48      Finance  127776.0           4.385456
    emp_id         name   department    salary  years_experience  \
0   

In [68]:
department_stats = pd.read_sql('''
select department,count(*) as employee_count, avg(salary) as avg_salary from employees
group by department
order by avg_salary desc
'''
,conn)
print(department_stats)

    department  employee_count     avg_salary
0  Engineering              26  111634.269231
1      Finance              37  100886.945946
2    Marketing              31   98538.612903
3        Sales              28   94671.892857
4           HR              28   87384.714286


Calculate average performance_score by department using .groupby()
Replicate Query 1 using ONLY Pandas (.loc[], .sort_values(), .head())
Write 150-200 words comparing SQL vs Pandas:
Key syntax differences (=, AND, ORDER BY vs ==, &, .sort_values())
When to use each tool
Which is better for datasets too large for memory? Why?

In [72]:
df = pd.read_sql('''
select * from employees
'''
,conn)
#print(department_stats)
print(df.groupby('department')['performance_score'].mean())

#get unique departments
departments = sorted(df['department'].unique())

results = []
for dept in departments:

    dept_scores = df.loc[df['department'] == dept, 'performance_score']


    avg = dept_scores.mean()
    results.append({'department': dept, 'avg_performance': avg})


avg_df = pd.DataFrame(results)
final_result = avg_df.sort_values(by='avg_performance', ascending=False).head(5)

print(final_result)


department
Engineering    3.134223
Finance        3.135685
HR             3.070768
Marketing      3.318534
Sales          2.928389
Name: performance_score, dtype: float64
    department  avg_performance
3    Marketing         3.318534
1      Finance         3.135685
0  Engineering         3.134223
2           HR         3.070768
4        Sales         2.928389


Write 150-200 words comparing SQL vs Pandas:
Key syntax differences (=, AND, ORDER BY vs ==, &, .sort_values())
When to use each tool
Which is better for datasets too large for memory? Why?


Comparision = in sql and == in pandas
AND operator and in SQL ,& IN PANDAS
Sorting ORDER BY in sql and .sort_values in pandas
SQL is better because it pre processes the data on the database itself.